In [ ]:
"""
B站弹幕采集脚本：根据 BV 号获取 cid，再优先拉取分段弹幕，旧 XML 弹幕接口作为兜底。
说明：B站页面显示的弹幕总量不等于旧 XML 接口一次可返回的数量；分段接口通常能拿到更多。
输出：data/raw_bilibili_danmu.csv
"""
import csv
import math
import time
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import List, Dict, Optional

import requests

ROOT = Path(__file__).resolve().parents[1]
OUT = ROOT / "data" / "raw_bilibili_danmu.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.bilibili.com/",
    # 如遇到403/风控，可将浏览器 Cookie 填在这里：
    # "Cookie": "SESSDATA=...; bili_jct=...;"
}

# 示例：把每个分区准备若干 BV 号。真实提交时替换为你们选定的视频。
VIDEOS = [
    {"partition": "学习区", "bvid": "BV1BiDaB1EEN"},
    {"partition": "生活区", "bvid": "BV1MDA6zHEFE"},
]


def get_video_info(bvid: str) -> Dict[str, str]:
    url = "https://api.bilibili.com/x/web-interface/view"
    r = requests.get(url, params={"bvid": bvid}, headers=HEADERS, timeout=15)
    r.raise_for_status()
    data = r.json()
    if data.get("code") != 0:
        raise RuntimeError(f"获取 cid 失败：{bvid}, 返回={data}")
    info = data["data"]
    stat = info.get("stat", {})
    pages = [
        {
            "cid": int(page["cid"]),
            "part": page.get("part", ""),
            "duration": int(page.get("duration", 0) or 0),
        }
        for page in info.get("pages", [])
        if page.get("cid")
    ]
    return {
        "cid": int(info["cid"]),
        "title": info.get("title", ""),
        "duration": int(info.get("duration", 0) or 0),
        "pages": pages,
        "view_count": stat.get("view", 0),
        "like_count": stat.get("like", 0),
        "coin_count": stat.get("coin", 0),
        "favorite_count": stat.get("favorite", 0),
        "reply_count": stat.get("reply", 0),
        "danmaku_count": stat.get("danmaku", 0),
    }


def get_cid(bvid: str) -> int:
    return int(get_video_info(bvid)["cid"])


def _read_varint(data: bytes, pos: int):
    result = 0
    shift = 0
    while pos < len(data):
        byte = data[pos]
        pos += 1
        result |= (byte & 0x7F) << shift
        if not byte & 0x80:
            return result, pos
        shift += 7
    raise ValueError("invalid protobuf varint")


def _skip_protobuf_field(data: bytes, pos: int, wire_type: int) -> int:
    if wire_type == 0:
        _, pos = _read_varint(data, pos)
        return pos
    if wire_type == 1:
        return pos + 8
    if wire_type == 2:
        length, pos = _read_varint(data, pos)
        return pos + length
    if wire_type == 5:
        return pos + 4
    raise ValueError(f"unsupported protobuf wire type: {wire_type}")


def _parse_danmaku_elem_content(elem: bytes) -> str:
    pos = 0
    while pos < len(elem):
        key, pos = _read_varint(elem, pos)
        field_no = key >> 3
        wire_type = key & 0x07
        if field_no == 7 and wire_type == 2:
            length, pos = _read_varint(elem, pos)
            return elem[pos:pos + length].decode("utf-8", errors="ignore").strip()
        pos = _skip_protobuf_field(elem, pos, wire_type)
    return ""


def _parse_segment_reply(data: bytes) -> List[str]:
    """解析 x/v2/dm/web/seg.so 返回的 protobuf，提取 DanmakuElem.content。"""
    texts = []
    pos = 0
    while pos < len(data):
        key, pos = _read_varint(data, pos)
        field_no = key >> 3
        wire_type = key & 0x07
        if field_no == 1 and wire_type == 2:
            length, pos = _read_varint(data, pos)
            text = _parse_danmaku_elem_content(data[pos:pos + length])
            if text:
                texts.append(text)
            pos += length
        else:
            pos = _skip_protobuf_field(data, pos, wire_type)
    return texts


def get_danmu_xml(cid: int) -> List[str]:
    url = "https://api.bilibili.com/x/v1/dm/list.so"
    r = requests.get(url, params={"oid": cid}, headers=HEADERS, timeout=20)
    r.raise_for_status()
    r.encoding = "utf-8"
    root = ET.fromstring(r.text)
    return [node.text.strip() for node in root.findall("d") if node.text and node.text.strip()]


def get_danmu_segmented(cid: int, duration: int = 0) -> List[str]:
    url = "https://api.bilibili.com/x/v2/dm/web/seg.so"
    segment_count = max(1, math.ceil(duration / 360)) if duration else 1
    texts: List[str] = []
    for segment_index in range(1, segment_count + 1):
        r = requests.get(
            url,
            params={"type": 1, "oid": cid, "segment_index": segment_index},
            headers=HEADERS,
            timeout=20,
        )
        r.raise_for_status()
        texts.extend(_parse_segment_reply(r.content))
        time.sleep(0.08)
    return texts


def get_danmu(cid: int, duration: int = 0) -> List[str]:
    """优先用分段接口获取更完整弹幕；如果失败或数量更少，则回退到旧 XML 接口。"""
    segmented_texts: List[str] = []
    xml_texts: List[str] = []
    try:
        segmented_texts = get_danmu_segmented(cid, duration)
    except Exception as e:
        print(f"分段弹幕接口失败 cid={cid}: {e}")
    try:
        xml_texts = get_danmu_xml(cid)
    except Exception as e:
        print(f"XML 弹幕接口失败 cid={cid}: {e}")
    return segmented_texts if len(segmented_texts) >= len(xml_texts) else xml_texts


def crawl(
    videos: Optional[List[Dict[str, str]]] = None,
    out: Path = OUT,
    max_per_video: Optional[int] = None,
) -> List[Dict[str, str]]:
    videos = videos or VIDEOS
    out = Path(out)
    out.parent.mkdir(parents=True, exist_ok=True)
    rows: List[Dict[str, str]] = []
    for item in videos:
        bvid = item["bvid"]
        partition = item["partition"]
        try:
            video_info = get_video_info(bvid)
            texts: List[str] = []
            pages = video_info.get("pages") or [
                {"cid": video_info["cid"], "duration": video_info.get("duration", 0), "part": ""}
            ]
            for page in pages:
                texts.extend(get_danmu(int(page["cid"]), int(page.get("duration", 0) or 0)))
            fetched_count = len(texts)
            if max_per_video:
                texts = texts[:max_per_video]
            for text in texts:
                rows.append({
                    "partition": partition,
                    "bvid": bvid,
                    "title": video_info["title"],
                    "view_count": video_info["view_count"],
                    "like_count": video_info["like_count"],
                    "coin_count": video_info["coin_count"],
                    "favorite_count": video_info["favorite_count"],
                    "reply_count": video_info["reply_count"],
                    "danmaku_count": video_info["danmaku_count"],
                    "text": text,
                })
            limit_note = f"，按实验上限保留 {len(texts)} 条" if max_per_video and fetched_count > len(texts) else ""
            print(
                f"{partition} {bvid}: 页面弹幕数 {video_info['danmaku_count']}，"
                f"实际获取 {fetched_count} 条{limit_note}"
            )
            if video_info["danmaku_count"] and fetched_count < int(video_info["danmaku_count"]):
                print(
                    "提示：页面弹幕数通常包含累计/历史弹幕，公开接口可能只返回当前可访问弹幕；"
                    "如需更多数据，可换弹幕更多且接口可返回更多的视频，或配置 Cookie 后扩展历史弹幕采集。"
                )
            time.sleep(1)
        except Exception as e:
            print(f"跳过 {bvid}: {e}")

    with out.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "partition", "bvid", "title", "view_count", "like_count",
            "coin_count", "favorite_count", "reply_count", "danmaku_count", "text",
        ])
        writer.writeheader()
        writer.writerows(rows)
    print(f"已保存：{out}, 共 {len(rows)} 条")
    return rows


if __name__ == "__main__":
    crawl()
